# 04 — Signal Development & Validation

Validates the `MomentumSignal` module and inspects signal output.

**EDA findings baked in:**
- Primary lookback: **8h** (Sharpe 1.94, best L/S spread)
- Secondary blend: **4h** (30% weight)
- Universe avg AC(1) = -0.007 → near-zero, cross-sectional approach correct
- Position sizing: **inverse-ATR volatility weighted**
- Max 10 positions, max 25% per coin

In [1]:
import sys
sys.path.insert(0, '..')   # allow imports from project root

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta

from signals.momentum import MomentumSignal, MomentumConfig, SignalDiagnostics, load_universe_from_csv

DB_PATH = '../data/ohlcv_1m.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)

# Load universe saved by EDA notebook
universe = load_universe_from_csv('../data/tradeable_universe.csv')
print(f'Universe: {len(universe)} coins')

Universe: 58 coins


## 1. Instantiate Signal & Inspect Latest Output

In [2]:
cfg = MomentumConfig(
    lookback_primary   = 8,
    lookback_secondary = 4,
    weight_primary     = 0.70,
    weight_secondary   = 0.30,
    top_n              = 10,
    max_weight         = 0.25,
    vol_lookback       = 14,
)

signal = MomentumSignal(con=con, universe=universe, cfg=cfg)
diag   = SignalDiagnostics(signal)

# Use latest available timestamp in DB (rather than now())
latest_ts = con.execute("""
    SELECT MAX(ts) FROM ohlcv WHERE interval = '1m'
""").fetchone()[0]
latest_ts = pd.to_datetime(latest_ts, utc=True).replace(minute=0, second=0, microsecond=0)
print(f'Computing signal as of: {latest_ts}')

# Full score table
score_table = diag.score_table(latest_ts)
print(f'\nSignal output — {len(score_table)} coins scored:')
display(score_table[['mom_8h','mom_4h','score','rank','atr_pct_14h','weight','selected']]
        .style
        .background_gradient(subset=['score'], cmap='RdYlGn')
        .background_gradient(subset=['weight'], cmap='Blues')
        .format({'mom_8h':'{:.4f}','mom_4h':'{:.4f}','score':'{:.4f}',
                 'atr_pct_14h':'{:.3f}%','weight':'{:.4f}'}))

Computing signal as of: 2026-03-18 07:00:00+00:00

Signal output — 58 coins scored:


,mom_8h,mom_4h,score,rank,atr_pct_14h,weight,selected
XPL-USD,0.0521,0.0367,1.0000,1,2.174%,0.0749,True
BONK-USD,0.0259,0.0167,0.9655,2,1.480%,0.1100,True
FET-USD,0.0254,0.0301,0.9466,3,2.615%,0.0623,True
PENGU-USD,0.0287,0.0095,0.9414,4,1.385%,0.1176,True
APT-USD,0.0189,0.0139,0.8828,5,1.001%,0.1626,True
WLD-USD,0.0257,0.0050,0.8810,6,1.210%,0.1346,True
DOT-USD,0.0186,0.0136,0.8655,7,0.858%,0.1898,True
S-USD,0.0209,0.0057,0.8603,8,0.907%,0.0000,True
LINEA-USD,0.0170,0.0142,0.8517,9,1.107%,0.0000,True
ENA-USD,0.0153,0.0119,0.8241,10,1.099%,0.1482,True


In [ ]:
# Portfolio weight bar chart
selected = score_table[score_table['selected'] & (score_table['weight'] > 0)].copy()
selected = selected.sort_values('weight', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(selected['score'].values)
bars = ax.barh(selected.index.str.replace('-USD',''), selected['weight'] * 100,
               color=colors, edgecolor='white', height=0.7)
ax.set_xlabel('Portfolio Weight (%)')
ax.set_title(f'Signal Weights as of {latest_ts.strftime("%Y-%m-%d %H:%M UTC")}', fontweight='bold')
ax.axvline(100/len(selected), color='gray', linestyle='--', linewidth=1, label='Equal weight')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nTotal allocated: {selected["weight"].sum()*100:.1f}%')
print(f'Cash buffer    : {(1 - selected["weight"].sum())*100:.1f}%')

## 2. Walk-Forward Signal Validation

Replay signal across all historical hourly timestamps.
Check:
- Weight stability (turnover per rebalance)
- Signal concentration over time
- How often min_n threshold is not met

In [ ]:
# Generate hourly timestamps over the data window
first_ts = con.execute("SELECT MIN(ts) FROM ohlcv WHERE interval = '1m'").fetchone()[0]
first_ts = pd.to_datetime(first_ts, utc=True).replace(minute=0, second=0, microsecond=0)

# Start after enough warmup (need at least lookback_primary + vol_lookback hours)
warmup_h = cfg.lookback_primary + cfg.vol_lookback + 5
start_ts = first_ts + timedelta(hours=warmup_h)

timestamps = pd.date_range(start=start_ts, end=latest_ts, freq='1h', tz='UTC').tolist()
print(f'Replaying signal over {len(timestamps)} hourly timestamps...')
print(f'From: {timestamps[0]}  →  {timestamps[-1]}')

In [ ]:
# Batch compute — this may take 1-3 minutes depending on data size
weights_df = signal.compute_batch(timestamps)
print(f'Weight matrix shape: {weights_df.shape}')
print(f'Non-zero positions per row: {(weights_df > 0).sum(axis=1).describe().round(2)}')

In [ ]:
# Turnover analysis
# Turnover = sum(|w_t - w_{t-1}|) / 2 per rebalance
turnover = weights_df.diff().abs().sum(axis=1) / 2
turnover = turnover.dropna()

print(f'Turnover per rebalance (1h):')
print(f'  Mean   : {turnover.mean()*100:.2f}%')
print(f'  Median : {turnover.median()*100:.2f}%')
print(f'  95th % : {turnover.quantile(0.95)*100:.2f}%')
print()
print(f'Estimated fee per rebalance (0.10% taker × turnover):')
print(f'  Mean   : {turnover.mean() * 0.001 * 100:.4f}%')
print(f'  Daily  : {turnover.mean() * 0.001 * 24 * 100:.3f}% (24 rebalances/day)')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
turnover.plot(ax=axes[0], color='#3498db', linewidth=0.8)
axes[0].set_title('Turnover per Rebalance', fontweight='bold')
axes[0].set_ylabel('Turnover (fraction of portfolio)')
turnover.plot.hist(bins=50, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Turnover Distribution', fontweight='bold')
axes[1].set_xlabel('Turnover')
plt.tight_layout()
plt.show()

In [ ]:
# Weight stability heatmap — which coins are selected over time?
# Show top 20 most-selected coins
selection_freq = (weights_df > 0).mean().sort_values(ascending=False)
top20 = selection_freq.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(18, 6))
heat_data = weights_df[top20].T
im = ax.imshow(heat_data.values, aspect='auto', cmap='YlOrRd',
               vmin=0, vmax=cfg.max_weight)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([s.replace('-USD','') for s in top20], fontsize=8)
# Only show every 24th timestamp on x-axis
step = max(1, len(heat_data.columns) // 20)
ax.set_xticks(range(0, len(heat_data.columns), step))
ax.set_xticklabels(
    [heat_data.columns[i].strftime('%m-%d') for i in range(0, len(heat_data.columns), step)],
    rotation=45, fontsize=7
)
plt.colorbar(im, ax=ax, label='Weight')
ax.set_title('Portfolio Weight Heatmap — Top 20 Coins Over Time', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/signal_weight_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSelection frequency (% of hours in portfolio):')
print(selection_freq.head(20).mul(100).round(1).to_string())

## 3. Signal Sensitivity Analysis

Test how signal quality changes across different lookback windows.
Confirm 8h is robust and not just a lucky parameter.

In [ ]:
# Quick sensitivity: vary lookback_primary from 4 to 24h
# Measure: avg top-quintile forward return (proxy for signal quality)

pivot_close = con.execute("""
    SELECT DATE_TRUNC('hour', ts) AS ts_hour, symbol,
           LAST(close ORDER BY ts) AS close
    FROM ohlcv
    WHERE interval = '1m'
    GROUP BY DATE_TRUNC('hour', ts), symbol
    ORDER BY ts_hour
""").df()

pivot_close['ts_hour'] = pd.to_datetime(pivot_close['ts_hour'], utc=True)
pivot = pivot_close[pivot_close['symbol'].isin(universe)].pivot_table(
    index='ts_hour', columns='symbol', values='close'
).sort_index()

sensitivity_results = []
for lb in [2, 4, 6, 8, 10, 12, 16, 20, 24]:
    mom  = pivot.pct_change(lb)
    fwd  = pivot.pct_change(1).shift(-1)
    rnk  = mom.rank(axis=1, pct=True)
    top  = fwd.where(rnk >= 0.8).mean(axis=1).dropna()
    sensitivity_results.append({
        'lookback_h'   : lb,
        'mean_fwd_ret' : top.mean() * 100,
        'sharpe'       : top.mean() / top.std() * np.sqrt(24 * 365) if top.std() > 0 else 0,
        'hit_rate'     : (top > 0).mean() * 100,
    })

sens_df = pd.DataFrame(sensitivity_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (col, label) in enumerate([('mean_fwd_ret','Avg Forward Return (%)'),
                                    ('sharpe','Annualised Sharpe'),
                                    ('hit_rate','Hit Rate (%)')]):
    colors = ['#e74c3c' if lb == 8 else '#3498db' for lb in sens_df['lookback_h']]
    axes[i].bar(sens_df['lookback_h'].astype(str), sens_df[col], color=colors, edgecolor='white')
    axes[i].set_xlabel('Lookback (hours)')
    axes[i].set_ylabel(label)
    axes[i].set_title(label, fontweight='bold')
    axes[i].axhline(0, color='black', linewidth=0.8)

plt.suptitle('Signal Sensitivity to Lookback Parameter\n(Red = chosen 8h)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/signal_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print(sens_df.round(4).to_string(index=False))

In [ ]:
con.close()
print('Done. Next: research/05_backtest.ipynb')